In [ ]:
import pydicom
from pathlib import Path
import pandas as pd
from tqdm import tqdm

main_folder = Path(r"\\pom\pmcommon\_PM.Private\CSIC\Moulin\4D Flow Data")

#General target series description, change to specific series description if needed
TARGET_SERIES_DESCRIPTION = "4D"
MATCH_MODE = "contains"

records = []
no_match = []

# Get all existing patient directories from the main folder
patient_dirs = [d for d in main_folder.iterdir() if d.is_dir()]

for patient_dir in tqdm(patient_dirs):
    found_any = False
    for sequence_dir in sorted(patient_dir.iterdir()):
        if not sequence_dir.is_dir():
            continue

        dcm_files = list(sequence_dir.glob("*.dcm"))
        if not dcm_files:
            continue

        try:
            # Read the first DICOM file in the sequence to check the SeriesDescription
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            series_desc = getattr(ds, "SeriesDescription", "") or ""

            match = (
                TARGET_SERIES_DESCRIPTION.lower() in series_desc.lower()
                if MATCH_MODE == "contains" 
                else series_desc.lower() == TARGET_SERIES_DESCRIPTION.lower()
            )

            # If the series description matches, store the relevant information
            if match:
                records.append({
                    "patient_folder": patient_dir.name,
                    "sequence_folder": sequence_dir.name,
                    "num_files": len(dcm_files),
                    "PatientMRN": getattr(ds, "PatientID", None),
                    "SeriesDescription": series_desc,
                    "Modality": getattr(ds, "Modality", None),
                })
                found_any = True
        except Exception as e:
            print(f"Skipped {sequence_dir}: {e}")
            continue

        if not found_any:
            no_match.append(patient_dir.name)

df = pd.DataFrame(records)
print(f"Found {len(df)} matching sequences across {df['patient_folder'].nunique()} patients")
print(f"{len(no_match)} patients had no matching sequence")


100%|██████████| 141/141 [29:06<00:00, 12.39s/it]

Found 410 matching sequences across 139 patients
4989 patients had no matching sequence


In [2]:
# Group rows by patient folder name
df_sorted = df.sort_values("patient_folder").reset_index(drop=True)
df_sorted

df_sorted.to_csv('4DFlowDisplay.csv')